<a href="https://colab.research.google.com/github/crystalloide/Big-Data-Cluster/blob/main/tp_hive_iceberg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP – Création d’une table Iceberg avec Hive

Objectifs du TP :
- Démarrer un environnement Hive capable de gérer des tables Iceberg.
- Créer une base `nyc` et une table `taxis` stockée au format Iceberg.
- Insérer et requêter des données pour vérifier le fonctionnement.

Ce TP suit la syntaxe du quickstart Hive + Iceberg officiel. [web:18][web:22]


## 1. Préparation de l’environnement Hive + Iceberg

Ce TP suppose :
- Hive 4.0.0+ (Iceberg intégré) ou Hive 3.1.x avec les JARs Iceberg ajoutés.
- Un warehouse HDFS accessible (ex : `hdfs://namenode:8020/user/hive/warehouse`).
- Accès à Hive via Beeline ou un moteur SQL depuis ce notebook (par exemple un kernel avec magics `%sql`). [web:18][web:22]

Si tu utilises un cluster Docker de démo (optionnel), tu peux lancer un conteneur HiveServer2.


In [ ]:
%%bash
# (Optionnel) Exemple de lancement rapide d'un HiveServer2 en Docker
# À adapter selon ton environnement. Peut être ignoré si Hive est déjà disponible.

HIVE_VERSION=4.0.0

docker run -d \
  -p 10000:10000 -p 10002:10002 \
  --env SERVICE_NAME=hiveserver2 \
  --name hive4 \
  apache/hive:${HIVE_VERSION}

docker ps | grep hive4 || echo "HiveServer2 non démarré (adapter la commande Docker si nécessaire)."


## 2. Connexion depuis le notebook (exemple PySpark + Hive)

Si tu disposes d'un environnement PySpark avec support Hive, tu peux initialiser une `SparkSession` avec Hive. [web:22]

Sinon, passe directement aux cellules SQL (via un client Hive externe ou des magics `%sql`).


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("HiveIcebergTP")
    .enableHiveSupport()
    .config("spark.sql.warehouse.dir", "hdfs://namenode:8020/user/hive/warehouse")
    .getOrCreate()
)

spark.sql("SHOW DATABASES").show()


Si tu utilises un client Hive direct (Beeline, etc.), tu peux ignorer la cellule PySpark et te concentrer sur les commandes SQL suivantes.


## 3. Configuration du catalog Iceberg côté Hive

On configure un catalog Iceberg basé sur HDFS (HadoopCatalog). [web:22]

En Hive 3.1.x, il peut être nécessaire d'ajouter le JAR Iceberg dans la session (`ADD JAR`). [web:22]


In [ ]:
%%sql
-- Si tu es en Hive 3.1.x sans intégration Iceberg, décommente la ligne suivante avec le bon chemin :
-- ADD JAR /opt/iceberg/iceberg-hive-runtime-1.5.1.jar;

-- Déclaration d'un catalog Iceberg basé sur HDFS
SET iceberg.catalog.hadoop.type=hadoop;
SET iceberg.catalog.hadoop.warehouse=hdfs://namenode:8020/user/hive/warehouse/iceberg_warehouse;


## 4. Création de la base de travail `nyc`

On suit la convention du quickstart Iceberg/Hive qui utilise une base `nyc`. [web:18]


In [ ]:
%%sql
CREATE DATABASE IF NOT EXISTS nyc;
USE nyc;

SHOW DATABASES;


## 5. Création d’une table Iceberg `nyc.taxis`

Nous créons une table Iceberg partitionnée par `vendor_id`. Syntaxe inspirée directement du quickstart Iceberg Hive. [web:18][web:22]


In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS nyc.taxis (
  trip_id            BIGINT,
  trip_distance      FLOAT,
  fare_amount        DOUBLE,
  store_and_fwd_flag STRING
)
PARTITIONED BY (vendor_id BIGINT)
STORED BY ICEBERG;


Selon ta version de Hive/Iceberg, tu peux aussi expliciter le `StorageHandler` Iceberg. [web:22]


In [ ]:
%%sql
-- Variante alternative avec StorageHandler explicite
-- À utiliser seulement si ta distribution le recommande.

CREATE TABLE IF NOT EXISTS nyc.taxis_sh (
  trip_id            BIGINT,
  trip_distance      FLOAT,
  fare_amount        DOUBLE,
  store_and_fwd_flag STRING
)
PARTITIONED BY (vendor_id BIGINT)
STORED BY 'org.apache.iceberg.mr.hive.HiveIcebergStorageHandler'
TBLPROPERTIES ('iceberg.catalog' = 'hadoop');


## 6. Insertion de données de test

On insère quelques lignes directement dans la table Iceberg pour vérifier les DML. [web:22]


In [ ]:
%%sql
INSERT INTO nyc.taxis VALUES
  (1, 1.5, 10.0, 'N', 1),
  (2, 3.2, 18.5, 'N', 2),
  (3, 0.8,  6.0, 'Y', 1);


## 7. Lecture et vérification de la table Iceberg

On lit les données et on inspecte les métadonnées pour confirmer que la table est bien gérée par Iceberg. [web:22]


In [ ]:
%%sql
SELECT * FROM nyc.taxis;


In [ ]:
%%sql
DESCRIBE FORMATTED nyc.taxis;


Tu devrais voir :
- `Table Type: MANAGED_TABLE`.
- Un StorageHandler Iceberg (`STORED BY ICEBERG` ou `HiveIcebergStorageHandler`).
- Des propriétés Iceberg dans les TBLPROPERTIES. [web:22]


## 8. Extensions possibles

Idées pour enrichir le TP dans un contexte Lakehouse :
- Recréer la table avec un partitionnement plus avancé (par date, bucket, etc.). [web:22]
- Tester un `ALTER TABLE nyc.taxis ADD COLUMNS (tip_amount DOUBLE);` puis réinsérer des lignes et observer le schema evolution. [web:22]
- Monter un catalog Iceberg partagé et lire la même table depuis Spark ou Trino, pour illustrer la séparation compute / storage. [web:22]
